In [19]:
import numpy as np
import scipy.stats as stats
from scipy.optimize import minimize, curve_fit

Consider the problem of obtaining a relationship for the ratio between the
equilibrium and isotonic cell volumes given the osmotic pressure. The
theoretical relationship can be written as

$$
\frac{V}{V_0}
=
(1 - b^*)
\frac{-1 + \sqrt{1 + 4 B \,\Pi_0}}
     {-1 + \sqrt{1 + 4 B \,\Pi}}
+ b^*,
\tag{21}
$$

where both $B$ and $b^*$ are parameters to be determined, and $\Pi_0$ is a
known osmotic value. In this example, $\Pi_0$ has the value $0.293$.


In [17]:
import numpy as np

# Data x and measured values y
x = np.array([
    0.29278, 0.57172, 0.85514, 1.13595, 1.43349,
    1.72908, 2.02815, 2.3266, 2.66704
])

y = np.array([
    1.00034, 0.80465, 0.75358, 0.71548, 0.68588,
    0.666, 0.65913, 0.64004, 0.62661
])

Pi0 = 0.293  # known osmotic value

alpha = 0.05 # significance level

In [4]:
def srs(beta):
    """
    Computes the sum of squared residuals for a given parameter vector beta.
    """
    y_hat = model(x, beta)
    residuals = y - y_hat
    return np.sum(residuals**2)

In [3]:
def model(x, beta):
    """
    Computes predicted values y_hat given regressors x and parameter vector beta.

    beta[0] = b*
    beta[1] = B
    """

    b_star = beta[0]
    B = beta[1]

    num = -1 + np.sqrt(1 + 4 * B * Pi0)
    den = -1 + np.sqrt(1 + 4 * B * x)

    return (1 - b_star) * (num / den) + b_star

As an initial guess for the parameter estimates we take
$b^* = 0.2$ and $B = 0.56$. To obtain the nonlinear
least-squares estimates we minimise the sum of squared residuals
with respect to the parameter vector $\beta = [b^*, B]^T$.

In [6]:
# Initial guess: b* = 0.2, B = 0.56
beta0 = np.array([0.2, 0.56])

# Nelder–Mead in SciPy is the analogue of MATLAB's fminsearch
res = minimize(srs, beta0, method="Nelder-Mead")

betanlr = res.x

print("Estimated parameters (nonlinear regression):")
print(f"b* = {betanlr[0]:.4f}")
print(f"B  = {betanlr[1]:.4f}")
print("Function value SRS =", srs(betanlr))


Estimated parameters (nonlinear regression):
b* = 0.5246
B  = 2.4081
Function value SRS = 0.0003334376851234979


In order to compute further quantities, the derivatives of the
dimensionless volume ratio with respect to the parameters $b^*$ and $B$ are required.
The derivative with respect to $b^*$ is

$$
\frac{\partial}{\partial b^*}\left(\frac{V}{V_0}\right)
=
1
-
\frac{-1 + \sqrt{1 + 4 B \Pi_0}}
     {-1 + \sqrt{1 + 4 B \Pi}}.
$$

The derivative with respect to $B$ is

$$
\frac{\partial}{\partial B}\left(\frac{V}{V_0}\right)
=
2(1 - b^*)
\left[
\frac{\Pi_0}
     {\sqrt{1 + 4 B \Pi_0}\,\bigl(-1 + \sqrt{1 + 4 B \Pi}\bigr)}
-
\frac{\Pi\bigl(-1 + \sqrt{1 + 4 B \Pi_0}\bigr)}
     {\sqrt{1 + 4 B \Pi}\,\bigl(-1 + \sqrt{1 + 4 B \Pi}\bigr)^2}
\right].
$$


In [7]:
def derivatives(x, beta):
    """
    Computes partial derivatives of the model output with respect to
    the parameters beta = [b*, B].

    Parameters
    ----------
    x : array_like values (same x as in the model function).
    beta : array_like, shape (2,) parameter vector [b_star, B].

    Returns
    -------
    yderiv : ndarray, shape (len(x), 2)
        Column 0: d(V/V0)/db*
        Column 1: d(V/V0)/dB
    """
    x = np.asarray(x, dtype=float)
    b_star, B = beta

    # Common components (match MATLAB s0 and s)
    s0 = np.sqrt(1 + 4 * B * Pi0)   # scalar
    s = np.sqrt(1 + 4 * B * x)      # array, same shape as x

    # Derivative w.r.t. b*
    dy_db = 1 - (s0 - 1) / (s - 1)

    # Derivative w.r.t. B
    dy_dB = 2 * (1 - b_star) * (
        Pi0 / s0 / (s - 1) - x * (s0 - 1) / s / (s - 1)**2
    )

    # Stack as columns: (N, 2) just like MATLAB's yderiv(:,1), yderiv(:,2)
    yderiv = np.column_stack((dy_db, dy_dB))
    return yderiv

In [10]:
# Identifying the values
m=9
n=2
gJ=derivatives(x,betanlr)

In [11]:
# Estimating outputs
yhat=model(x,betanlr)

In [12]:
# Residuals
resid=yhat-y

In [15]:
# Standart deviation
sig_mod = np.sqrt(np.sum(resid**2) / (m - n))

print("Residuals:", resid)
print(f"Model standard deviation (sig_mod) = {sig_mod:.4f}")

Residuals: [-7.00314476e-05  1.27037353e-02 -6.33978082e-03 -5.99997459e-03
 -1.87460198e-03  3.94051855e-04 -5.98113734e-03  2.83257912e-03
  6.95618122e-03]
Model standard deviation (sig_mod) = 0.0069


In [16]:
# Parameter covariance approximation
F = np.linalg.inv(gJ.T @ gJ)

In [18]:
# Half–widths of the confidence intervals (deltas)
delta1 = stats.t.ppf(1 - alpha/2, m-n) * sig_mod * np.sqrt(F[0, 0])
delta2 = stats.t.ppf(1 - alpha/2, m-n) * sig_mod * np.sqrt(F[1, 1])

print(f"delta1 = {delta1:.4f}")
print(f"delta2 = {delta2:.4f}")

delta1 = 0.0436
delta2 = 3.6163


To use a built-in functions, the model will once again be defined

In [20]:
def modelnl_curvefit(x, b_star, B):
    """
    Nonlinear model: V/V0 as a function of x, with parameters b* (b_star) and B.
    """
    x = np.asarray(x, dtype=float)

    num = -1 + np.sqrt(1 + 4 * B * Pi0)
    den = -1 + np.sqrt(1 + 4 * B * x)

    return (1 - b_star) * (num / den) + b_star

Nonlinear least squares: analogue of 'nlinfit'

In [21]:
# Initial guess: b* = 0.2, B = 0.56
p0 = [0.2, 0.56]

# curve_fit is the rough analogue of nlinfit
popt, pcov = curve_fit(modelnl_curvefit, x, y, p0=p0)

betanl = popt  # [b* , B]

print("Parameter estimates (betanl):")
print(f"b* = {betanl[0]:.4f}")
print(f"B  = {betanl[1]:.4f}")

Parameter estimates (betanl):
b* = 0.5246
B  = 2.4081


In [23]:
yhat = modelnl_curvefit(x, *betanl)
r = y - yhat

print("Residuals r:", r)

Residuals r: [ 7.00313930e-05 -1.27037301e-02  6.33976474e-03  5.99993934e-03
  1.87455006e-03 -3.94117290e-04  5.98106056e-03 -2.83266545e-03
 -6.95627679e-03]


In [22]:
# Standard errors of the parameters from covariance matrix
se = np.sqrt(np.diag(pcov))

# critical value for (1 - alpha) CI
tcrit = stats.t.ppf(1 - alpha/2, m-n)

# half-widths of the confidence intervals (deltas)
delta = tcrit * se

# 2×2 matrix of CIs: rows = parameters, columns = [lower, upper]
CI = np.column_stack((betanl - delta, betanl + delta))

print("95% confidence intervals:")
print(f"CI for b* : [{CI[0,0]:.4f}, {CI[0,1]:.4f}]")
print(f"CI for B  : [{CI[1,0]:.4f}, {CI[1,1]:.4f}]")

95% confidence intervals:
CI for b* : [0.4810, 0.5682]
CI for B  : [-1.2081, 6.0244]
